In [1]:
from __future__ import annotations

import os
from dataclasses import dataclass

import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
from astropy.coordinates import SkyCoord
import astropy.units as u
from joblib import Parallel, delayed
from scipy.stats import chi2 as chi2_dist
from scipy.stats import norm
import treecorr
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib

In [ ]:
# ==============================================================
# 1. General configuration
# ==============================================================

BASE_PATH = "/home/brunowesley/projetos/FRB-isotropy-tests/FRB_catalogs/"
ALL_FRB_PATH = os.path.join(BASE_PATH, "SkyPosition.csv")

N_RAND_FACTOR = 20
N_MOCKS = 200
N_ENSEMBLE = 10
N_MOCKS_PER_ENSEMBLE = max(1, N_MOCKS // N_ENSEMBLE)

GAL_CUT = 20.0

NSIDE_SF = 64
SMOOTH_SIGMA = 5.0
PERTURBATION_SCALE = 1.0

NSIDE_SF_RANGE = [32, 64, 128]
SMOOTH_SIGMA_RANGE = [3.0, 5.0, 8.0, 10.0]
N_SENSITIVITY_MOCKS = 50

OVERLAP_RADIUS_DEG = 5.0
OVERLAP_NSIDE = 32

USE_LOG = False
MAX_SEP = 180.0
MIN_SEP = 0.1

if USE_LOG:
    BIN_TYPE = "Log"
    N_BINS = 20
else:
    BIN_TYPE = "Linear"
    BIN_SIZE = 1.8
    N_BINS = int((MAX_SEP - MIN_SEP) / BIN_SIZE)

COARSE_BINS = np.arange(0, 181, 20)
COARSE_CENTERS = 0.5 * (COARSE_BINS[:-1] + COARSE_BINS[1:])


@dataclass
class TestStatistics:
    chi2: float
    chi2_red: float
    p_chi2: float
    p_empirical: float
    sigma_equiv: float
    global_tension: float
    abs_observed_stat: float
    abs_empirical_p: float
    hartlap_factor: float

In [ ]:
# ==============================================================
# 2. Catalog loading and mask
# ==============================================================

def load_catalog(path: str = ALL_FRB_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("/", "_")
    )
    return df[["RA", "DEC", "Reporting_Group_s"]].dropna().reset_index(drop=True)


def apply_mask(df: pd.DataFrame, gal_cut: float = GAL_CUT) -> pd.DataFrame:
    coords = SkyCoord(
        ra=df["RA"].values * u.degree,
        dec=df["DEC"].values * u.degree,
        frame="icrs",
    )
    b = coords.galactic.b.degree
    return df[np.abs(b) > gal_cut].reset_index(drop=True)


def healpix_galactic_mask(nside: int, gal_cut: float = GAL_CUT) -> np.ndarray:
    npix = hp.nside2npix(nside)
    theta, phi = hp.pix2ang(nside, np.arange(npix))
    coords = SkyCoord(
        ra=np.degrees(phi) * u.degree,
        dec=(90.0 - np.degrees(theta)) * u.degree,
        frame="icrs",
    )
    return np.abs(coords.galactic.b.degree) > gal_cut

In [ ]:
# ==============================================================
# 3. Survey split and selection functions
# ==============================================================

def split_by_survey(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    df = df.copy()
    df["Reporting_Group_s"] = df["Reporting_Group_s"].fillna("UNKNOWN")

    surveys: dict[str, list[tuple[float, float]]] = {}
    for row in df.itertuples(index=False):
        for group in str(row.Reporting_Group_s).split(","):
            group = group.strip()
            surveys.setdefault(group, []).append((row.RA, row.DEC))

    return {
        name: pd.DataFrame(values, columns=["RA", "DEC"])
        for name, values in surveys.items()
    }


def build_selection_function_improved(
    subdf: pd.DataFrame,
    nside: int = NSIDE_SF,
    smooth_sigma: float = SMOOTH_SIGMA,
    gal_cut: float = GAL_CUT,
) -> tuple[np.ndarray, np.ndarray]:
    """Build one survey SF plus a diagonal Poisson covariance estimate."""
    npix = hp.nside2npix(nside)

    theta = np.radians(90.0 - subdf["DEC"].values)
    phi = np.radians(subdf["RA"].values)
    pix = hp.ang2pix(nside, theta, phi)

    counts = np.bincount(pix, minlength=npix).astype(float)
    total_counts = counts.sum()
    if total_counts <= 0:
        raise ValueError("Cannot build a selection function from an empty survey.")

    sf_err_counts = np.sqrt(np.maximum(counts, 1.0))
    sf = counts / total_counts
    sf_cov_diag = (sf_err_counts / total_counts) ** 2

    if smooth_sigma > 0:
        sf = hp.smoothing(sf, sigma=np.radians(smooth_sigma))
        sf = np.clip(sf, 0.0, None)
        # Smoothing induces pixel correlations; keep a conservative diagonal model.
        sf_cov_diag *= 2.0

    gal_mask = healpix_galactic_mask(nside, gal_cut=gal_cut)
    sf[~gal_mask] = 0.0

    sf_sum = sf.sum()
    if sf_sum <= 0:
        raise ValueError("Selection function vanished after masking.")
    sf /= sf_sum

    return sf, sf_cov_diag


def build_survey_selection_functions_improved(
    df: pd.DataFrame,
    nside: int = NSIDE_SF,
    smooth_sigma: float = SMOOTH_SIGMA,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray], dict[str, float], int]:
    surveys = split_by_survey(df)
    total_memberships = sum(len(subdf) for subdf in surveys.values())

    sf_dict: dict[str, np.ndarray] = {}
    sf_cov_diag_dict: dict[str, np.ndarray] = {}
    survey_weights: dict[str, float] = {}

    for name, subdf in surveys.items():
        sf, sf_cov_diag = build_selection_function_improved(
            subdf,
            nside=nside,
            smooth_sigma=smooth_sigma,
        )
        sf_dict[name] = sf
        sf_cov_diag_dict[name] = sf_cov_diag
        survey_weights[name] = len(subdf) / total_memberships

    return sf_dict, sf_cov_diag_dict, survey_weights, nside


def generate_sf_variant(
    sf_dict: dict[str, np.ndarray],
    sf_cov_diag_dict: dict[str, np.ndarray],
    perturbation_scale: float = PERTURBATION_SCALE,
    seed: int | None = None,
) -> dict[str, np.ndarray]:
    """Perturb each SF inside its diagonal Poisson uncertainty model."""
    rng = np.random.default_rng(seed)
    sf_dict_perturbed: dict[str, np.ndarray] = {}

    for survey_name, sf in sf_dict.items():
        sigma = np.sqrt(sf_cov_diag_dict[survey_name])
        noise = rng.normal(0.0, perturbation_scale * sigma)
        sf_pert = np.clip(sf + noise, 0.0, None)

        if sf_pert.sum() <= 0:
            sf_pert = sf.copy()
        else:
            sf_pert /= sf_pert.sum()

        sf_dict_perturbed[survey_name] = sf_pert

    return sf_dict_perturbed